In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"
GOLD_PATH = "abfss://gold@pravdatalake.dfs.core.windows.net"
GOLD_TABLE_PATH = f"{GOLD_PATH}/fact_sales"
GOLD_TABLE_NAME = "vehicle_sales.gold.fact_sales"

In [0]:
silver_sales = spark.read.format("delta").load(f"{SILVER_PATH}/sales_table")

In [0]:
dim_genmodel = spark.read.format("delta").load(f"{GOLD_PATH}/dim_genmodel")

In [0]:
dimension_date = spark.read.format("delta").load(f"{GOLD_PATH}/dimension_date")

In [0]:
dim_genmodel_lookup = dim_genmodel.select("Genmodel_ID")

In [0]:
dim_date_lookup = dimension_date.filter((col("month") == 1) & (col("day") == 1)).select("date_key", "year")

In [0]:
fact_sales_updates = (
    silver_sales
    .join(
        dim_date_lookup,
        silver_sales["Year"] == dim_date_lookup["year"],
        how="left"
    )
    .withColumn("gold_updated_timestamp", current_timestamp())
    .select(
        silver_sales["Genmodel_ID"],
        silver_sales["Year"],
        dim_date_lookup["date_key"],
        silver_sales["Units_Sold"],
        col("gold_updated_timestamp")
    )
)

In [0]:
fact_sales_updates.display()

####Data Quality checks

In [0]:
row_count = fact_sales_updates.count()

In [0]:
duplicate_key_count = fact_sales_updates.groupBy("Genmodel_ID", "Year").count().filter("count > 1").count()

In [0]:
unresolved_genmodel_count = (
    fact_sales_updates.join(dim_genmodel_lookup, "Genmodel_ID", "left_anti").count()
)

In [0]:
print(f"row count: {row_count}")
print(f"duplicate (Genmodel_ID, Year) count: {duplicate_key_count}")
print(f"Genmodel_ID not found in dim_genmodel count: {unresolved_genmodel_count}")

In [0]:
assert duplicate_key_count == 0, "(Genmodel_ID, Year) should be unique in fact_sales"

In [0]:
if DeltaTable.isDeltaTable(spark, GOLD_TABLE_PATH):
 
    fact_sales_table = DeltaTable.forPath(spark, GOLD_TABLE_PATH)
 
    (fact_sales_table.alias("t")
        .merge(fact_sales_updates.alias("s"), "t.Genmodel_ID = s.Genmodel_ID AND t.Year = s.Year")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
 
else:
 
    fact_sales_updates.write \
        .format("delta") \
        .mode("overwrite") \
        .save(GOLD_TABLE_PATH)

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {GOLD_TABLE_NAME}
    USING DELTA
    LOCATION '{GOLD_TABLE_PATH}'
""")

In [0]:
spark.sql(f"OPTIMIZE {GOLD_TABLE_NAME} ZORDER BY (Genmodel_ID)")